# Musanga V4 — NB00: RGB / DSM / LAS / MS inventory and input audit

**Inherited from V3_1 NB00:** input auditing and source-data provenance. V3_1 has already approved the KVO field-grounded annotations; **do not reapprove, refit or replace them here.**

**Primary V4 hypothesis:** RGB + high-resolution DSM-derived structure improves held-out Musanga segmentation over RGB-only. LAS-derived 3D descriptors are a low-cost exploratory ablation only. No DTM, CHM, or under-canopy layer is assumed from SfM.

**Output contract:** `Final/Outputs/v4/00_input_audit/`. All earlier output directories and source rasters are read-only.

**Reproduction requirements:** same V3_1 30-circle fold membership (excluded original IDs 6, 9, 24, 25); preserve the 15 m spatial training exclusion, annotation semantics, and MOS 3–4 exclusion in the NB02B benchmark. This notebook **does not regenerate folds, train models, merge LAS, or process full point clouds**.

**Go/no-go checks:** confirm each RGB mosaic's intended DSM partner, exact CRS and coordinate transform, DSM valid coverage, SfM overlap, and acquisition chronology before enabling any V4 model-training notebook.

## 00 — Environment and read-only source registries

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas numpy matplotlib laspy
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import json, hashlib, re, warnings
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
from IPython.display import display
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT_CANDIDATES=[
    Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
    Path('/content/drive/MyDrive/Musanga/Final'),
]
ROOT=next((p for p in ROOT_CANDIDATES if p.exists()),None)
assert ROOT is not None, f'Could not find Final in {ROOT_CANDIDATES}'
DATA=ROOT/'Data'
assert DATA.exists(),f'Data is missing: {DATA}'
V31=ROOT/'Outputs/V3_1'
V31_MANIFEST=V31/'00_input_audit/00_input_manifest.json'
assert V31_MANIFEST.exists(),f'Approved V3_1 input manifest missing: {V31_MANIFEST}'
base=json.loads(V31_MANIFEST.read_text())
assert base.get('schema_version')=='v3_1_input_audit_kvo_final', base.get('schema_version')
OUT=ROOT/'Outputs/v4/00_input_audit'
OUT.mkdir(parents=True,exist_ok=True)
print('DATA (read-only):',DATA)
print('V3_1 manifest (read-only):',V31_MANIFEST)
print('V4 output:',OUT)
print('V3_1 annotation reference:',base.get('reference_version'))


## 01 — Discover RGB mosaics, DSM rasters and LAS without guessing site mappings

In [ ]:
FORESTS=('Baego','MOS','Yangambi','Yoko')
# Candidates only. Nothing is approved from filename matching alone.
SITE_KEYWORDS={
    'Baego': ('baego',),
    'MOS': ('mos_', 'mos-', 'mos1', 'mos5', 'mos 1', 'mos 5', 'mlt_mos'),
    'Yangambi': ('yangambi','carlier2020','ybjoseph','ybforgeo'),
    'Yoko': ('yoko',),
}
RGB_NAMES=[
    'Baego.tif',
    'Mos_1_2_transparent_mosaic_group1_georef.tif',
    'Mos_5_6_transparent_mosaic_group1_georef.tif',
    'Yangambi_georef.tif',
    'Yoko_kvo_georef.tif',
]
LAS_EXPECTED={
    'MOS': ['mlt_Mos_1_2_Green_densified_point_cloud_georef.las',
            'mlt_Mos_3_4_Green_densified_point_cloud_georef.las',
            'mlt_Mos_5_6_Green_densified_point_cloud_georef.las'],
    'Yangambi': ['Carlier2020_Aug20_flights_group1_densified_point_cloud_part_'+str(i)+'_georef.las' for i in range(1,7)]
              + ['YBJoseph_Green_densified_point_cloud_georef.las',
                 'YBForgeo_Green_densified_point_cloud_georef.las'],
    'Yoko': ['Yoko_survey_georef.las'],
    'Baego': [
        'Nadine_Baego_B2mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_B3mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_C2mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_C3mlt_2023_Green_densified_point_cloud.las',
    ],
}
# Bounded to project Data; metadata-only scan, no heavy raster/LAS loads.
paths=[p for p in DATA.rglob('*') if p.is_file() and p.suffix.lower() in ('.tif','.tiff','.las','.laz')]
rows=[]
for p in paths:
    name=p.name.lower()
    tags=[site for site, keys in SITE_KEYWORDS.items() if any(k in name or k in str(p.parent).lower() for k in keys)]
    rows.append({'path':str(p),'name':p.name,'type':'LAS' if p.suffix.lower() in ('.las','.laz') else 'raster',
                 'site_candidate':'|'.join(tags) if tags else '',
                 'dsm_candidate':bool(re.search(r'(^|[_\\-])(dsm|surface|elevation)([_\\-]|$)',p.stem,re.I)),
                 'bytes':p.stat().st_size})
inventory=pd.DataFrame(rows,columns=['path','name','type','site_candidate','dsm_candidate','bytes'])
inventory.to_csv(OUT/'00_file_inventory.csv',index=False)
display(inventory.groupby(['type','site_candidate'],dropna=False).size().rename('n').reset_index())
print('Rasters with DSM-style names:')
display(inventory.query('dsm_candidate').loc[:,['site_candidate','name','path']])
print('Known V3_1 active RGB mosaics:')
for name in RGB_NAMES:
    matches=inventory[inventory.name.str.lower()==name.lower()]
    print(name,'=>',matches.path.tolist())


## 02 — Fast raster metadata audit (no pixel reads)

Read TIFF headers only. Do not downsample massive UAV mosaics over Google Drive in NB00.


In [ ]:
def raster_info(path):
    # FAST: TIFF metadata only. No ds.read(), no resampling, no large Drive I/O.
    with rasterio.open(path) as ds:
        b=ds.bounds
        return {'path':str(path),'name':Path(path).name,
                'crs':str(ds.crs) if ds.crs else None,
                'width':ds.width,'height':ds.height,'bands':ds.count,
                'px_x':ds.transform.a,'px_y':ds.transform.e,
                'left':b.left,'bottom':b.bottom,'right':b.right,'top':b.top,
                'nodata':ds.nodata,'dtype':str(ds.dtypes[0]),
                'block_shapes':str(ds.block_shapes[:2]),
                'sample_valid_fraction':np.nan,
                'sample_z_min':np.nan,'sample_z_p50':np.nan,'sample_z_p99':np.nan}
raster_rows=[]
for p in inventory.query("type == 'raster'").path:
    try:
        raster_rows.append({**raster_info(p),'error':''})
    except Exception as exc:
        raster_rows.append({'path':p,'name':Path(p).name,'error':repr(exc)})
raster_df=pd.DataFrame(raster_rows)
raster_df.to_csv(OUT/'00_raster_metadata.csv',index=False)
display(raster_df[['name','crs','px_x','px_y','bands','width','height','error']])
print('Metadata audit complete. Valid-data fraction intentionally deferred to small local samples in NB01.')


## 03 — Explicit RGB–DSM file pairing and spatial-overlap audit

Use the five discovered DSMs. This audit checks metadata only: it does not certify within-forest subpixel alignment. In particular, MOS DSMs are coarser than their RGB orthomosaics.


In [ ]:
# Verified filenames from NB00's raster inventory. Locate exact paths in Data; no hardcoded directories.
DSM_BY_RGB={
    'Baego.tif': 'Baego_DSM.tif',
    'Mos_1_2_transparent_mosaic_group1_georef.tif': 'Mos_1_2_dsm_georef.tif',
    'Mos_5_6_transparent_mosaic_group1_georef.tif': 'mlt_Mos_5_6_dsm_georef (2).tif',
    'Yangambi_georef.tif': 'Yangambi_georef_DSM.tif',
    'Yoko_kvo_georef.tif': 'Yoko_kvo_georef_DSM.tif',
}
def intersection_fraction(a,b):
    xl=max(a[0],b[0]); yb=max(a[1],b[1]); xr=min(a[2],b[2]); yt=min(a[3],b[3])
    if xr<=xl or yt<=yb:return 0.
    return (xr-xl)*(yt-yb)/max((a[2]-a[0])*(a[3]-a[1]),1e-9)
overlap=[]
for rgb_name,dsm_name in DSM_BY_RGB.items():
    rgb_match=inventory[inventory.name.str.lower()==rgb_name.lower()]
    dsm_match=inventory[inventory.name.str.lower()==dsm_name.lower()]
    rec={'rgb':rgb_name,'rgb_matches':len(rgb_match),'dsm_name':dsm_name,
         'dsm_matches':len(dsm_match),'rgb_path':'','dsm_path':'',
         'crs_match':False,'same_grid':False,'overlap_fraction_of_rgb_bounds':np.nan,
         'extent_qa_pass':False,'registration_verified':False,'reason':''}
    if len(rgb_match)!=1 or len(dsm_match)!=1:
        rec['reason']='Missing/ambiguous RGB or DSM filename; inspect inventory'
    else:
        rec['rgb_path']=rgb_match.iloc[0].path
        rec['dsm_path']=dsm_match.iloc[0].path
        with rasterio.open(rec['rgb_path']) as rgb, rasterio.open(rec['dsm_path']) as dsm:
            if not rgb.crs or not dsm.crs:
                rec['reason']='Missing CRS'
            else:
                rec['crs_match']=bool(rgb.crs==dsm.crs)
                dsm_bounds=transform_bounds(dsm.crs,rgb.crs,*dsm.bounds,densify_pts=21)
                rec['overlap_fraction_of_rgb_bounds']=intersection_fraction(rgb.bounds,dsm_bounds)
                rec['same_grid']=bool(rgb.crs==dsm.crs and rgb.width==dsm.width
                                      and rgb.height==dsm.height
                                      and np.allclose(tuple(rgb.transform),tuple(dsm.transform),atol=1e-8,rtol=0))
                rec['extent_qa_pass']=bool(rec['overlap_fraction_of_rgb_bounds']>=0.95)
                rec['reason']=('Extent OK; check actual footprint and co-registration' if rec['extent_qa_pass']
                               else 'Extent overlap <95%; inspect intended survey coverage')
    overlap.append(rec)
pair_df=pd.DataFrame(overlap)
pair_df.to_csv(OUT/'00_rgb_dsm_pair_audit.csv',index=False)
display(pair_df[['rgb','dsm_name','crs_match','same_grid','overlap_fraction_of_rgb_bounds',
                 'extent_qa_pass','reason']])
print('Pairing metadata audit only; subpixel geometric registration and valid-pixel overlap are NOT yet verified.')
print('MOS DSM grid resolution differs from RGB; NB01 must use windowed resampling aligned to RGB.')
print('No training or mosaics were modified.')


## 04 — SfM LAS header-only audit, including Yangambi's six Carlier parts

In [ ]:
import laspy
las_rows=[]
for p in inventory.query("type == 'LAS'").path:
    try:
        with laspy.open(p) as lf:
            h=lf.header
            las_rows.append({'path':p,'name':Path(p).name,'point_count':int(h.point_count),
                             'crs':str(h.parse_crs()) if h.parse_crs() else '',
                             'xmin':float(h.mins[0]),'ymin':float(h.mins[1]),'zmin':float(h.mins[2]),
                             'xmax':float(h.maxs[0]),'ymax':float(h.maxs[1]),'zmax':float(h.maxs[2]),
                             'point_format':str(h.point_format.id),'version':str(h.version),'error':''})
    except Exception as exc:
        las_rows.append({'path':p,'name':Path(p).name,'error':repr(exc)})
las_df=pd.DataFrame(las_rows)
las_df.to_csv(OUT/'00_las_header_inventory.csv',index=False)
display(las_df)
expected=[(site,name) for site,names in LAS_EXPECTED.items() for name in names]
missing=[(site,name) for site,name in expected if name.lower() not in set(inventory.name.str.lower())]
print('Expected LAS missing from local Data/LAS:',missing)
print('Interpretation: SfM surface reconstruction, not penetrating lidar; do NOT infer DTM, CHM, or foliage profiles.')
print('Carlier parts 1–6 are one logical Yangambi dataset. Do not physically merge until CRS, coordinates and overlap pass.')


## 04B — Multispectral inventory and footprint metadata (V4 expansion)

**New optional modality, no V3_1 changes.** Inventory the actual `Final/Data/MS` folder only. Baego contains a 4-band raster (band wavelengths/order **not presumed**); MOS 1–2 and 5–6 contain green/red/red-edge/NIR TIFFs; Yangambi MS covers the **YBJoseph and YBForgeo right-hand survey acquisitions only**, **not** the left-hand Yangambi/Carlier survey. There is **no Yoko MS**. Distinguish survey/acquisition groups rather than assigning right-hand Yangambi MS to all Yangambi.

Header-only CRS, affine grid and rectangular intersection with the RGB reference are audited here. Actual valid overlap `ROI.shp ∩ RGB ∩ MS`, band calibration, band ordering, dates and image-level co-registration must be checked on small windows in NB01 before modelling. **Do not treat bounding-box overlap as an approved paired dataset.**

In [ ]:
# Header-only MS audit: cheap and bounded to Data/MS. Do not read any raster pixels here.
MS_DIR=DATA/'MS'
assert MS_DIR.is_dir(),f'Expected multispectral folder missing: {MS_DIR}'
ms_files=sorted(p for p in MS_DIR.rglob('*') if p.is_file() and p.suffix.casefold() in ('.tif','.tiff'))
def ms_group(p):
    n=p.name.casefold()
    if 'baego' in n:return 'Baego_MS'
    if 'mos_1_2' in n:return 'MOS_1_2_MS'
    if 'mos_5_6' in n:return 'MOS_5_6_MS'
    if 'ybjoseph' in n:return 'Yangambi_YBJoseph_right_MS'
    if 'ybforgeo' in n:return 'Yangambi_YBForgeo_right_MS'
    return 'UNASSIGNED'
def ms_band_hint(p):
    n=p.stem.casefold()
    if 'red edge' in n or 'red_edge' in n or 'rededge' in n:return 'red_edge'
    for band in ('green','nir','red'):
        if re.search(r'(^|[_ ])'+band+r'([_ ]|$)',n):return band
    return 'multi_band_unverified' if 'baego' in n else 'unassigned'
ms_meta=[]
for p in ms_files:
    try:
        with rasterio.open(p) as ds:
            b=ds.bounds
            ms_meta.append(dict(group=ms_group(p),band_hint=ms_band_hint(p),
                name=p.name,path=str(p),bands=ds.count,
                band_descriptions=str(ds.descriptions),
                crs=str(ds.crs) if ds.crs else '',
                width=ds.width,height=ds.height,
                pixel_x_m=float(ds.transform.a),pixel_y_m=float(ds.transform.e),
                left=b.left,bottom=b.bottom,right=b.right,top=b.top,
                nodata=ds.nodata,dtype=str(ds.dtypes[0]),error=''))
    except Exception as e:
        ms_meta.append(dict(group=ms_group(p),band_hint=ms_band_hint(p),
                            name=p.name,path=str(p),error=repr(e)))
MS_META=pd.DataFrame(ms_meta)
MS_META.to_csv(OUT/'00_ms_raster_headers.csv',index=False)
print('=== MS GROUPS / BANDS FOUND ===')
display(MS_META.groupby(['group','band_hint'],dropna=False).size().rename('files').reset_index())
display(MS_META[['group','band_hint','name','bands','band_descriptions','crs','error']])
expected={'Baego_MS','MOS_1_2_MS','MOS_5_6_MS','Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS'}
missing_groups=expected-set(MS_META.group)
print('Expected groups absent:',sorted(missing_groups))
print('Yoko: NO MS. Yangambi Carlier/left survey: NO MS.')
# Duplicate band types (e.g. MOS1_2 red-edge files) must be resolved, not selected by filename heuristics.
duplicates=MS_META[MS_META.band_hint.ne('multi_band_unverified') &
    MS_META.duplicated(['group','band_hint'],keep=False)]
if len(duplicates):
    print('AMBIGUOUS MS BANDS — inspect files; no automatic choice:')
    display(duplicates[['group','band_hint','name','path']])
# Extent overlap is descriptive only, NOT valid-data support or acquisition identity.
RGB_REFERENCE={
 'Baego_MS':'Baego.tif',
 'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
 'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
 'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
 'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
paired=[]
for group,rgb_name in RGB_REFERENCE.items():
    matches=inventory[inventory.name.str.casefold()==rgb_name.casefold()]
    assert len(matches)==1,('RGB reference ambiguous',rgb_name)
    with rasterio.open(matches.iloc[0].path) as rgb:
        for _,rec in MS_META[MS_META.group==group].iterrows():
            if rec.get('error',''):continue
            if not rec['crs']:
                frac=np.nan
            else:
                bb=transform_bounds(rec['crs'],rgb.crs,rec.left,rec.bottom,rec.right,rec.top,densify_pts=21)
                frac=intersection_fraction(rgb.bounds,bb)
            paired.append(dict(group=group,band_hint=rec.band_hint,name=rec['name'],
                rgb_reference=rgb_name,bbox_fraction_of_rgb=frac,
                rgb_crs=str(rgb.crs),ms_crs=rec.crs,
                same_crs=bool(str(rgb.crs)==rec.crs),
                acquisition_match_verified=False,roi_valid_overlap_verified=False))
MS_OVERLAP=pd.DataFrame(paired)
MS_OVERLAP.to_csv(OUT/'00_ms_rgb_extent_audit.csv',index=False)
display(MS_OVERLAP[['group','band_hint','name','bbox_fraction_of_rgb','same_crs']])
ms_registry=dict(schema='musanga_v4_ms_inventory_1',
    ms_dir=str(MS_DIR),
    groups={group:MS_META.loc[MS_META.group==group,['name','path','band_hint','bands']].to_dict('records')
            for group in sorted(MS_META.group.unique())},
    rgb_reference=RGB_REFERENCE,
    known_absence=['Yoko','Yangambi_Carlier_left'],
    notes=['Yangambi YBJoseph and YBForgeo are right-hand independent acquisitions',
           'Baego 4-band ordering and wavelengths require verification',
           'Band radiometry, date, ROI validity and co-registration not yet verified',
           'MS/DSM/LAS comparisons require matched valid ROI support'])
(OUT/'00_ms_registry.json').write_text(json.dumps(ms_registry,indent=2))
print('Saved MS header audit, extent audit and registry. No large raster reads performed.')


## 05 — Freeze proposed V4 audit contract (no modelling yet)

In [ ]:
audit={
    'schema_version':'v4_nb00_rgb_dsm_sfm_inventory_1',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'inherited_reference_manifest':str(V31_MANIFEST),
    'inherited_reference_schema':base.get('schema_version'),
    'inherited_reference_version':base.get('reference_version'),
    'v3_1_folds_policy':'Reuse frozen 30-circle within-site spatial folds; do not regenerate',
    'v3_1_excluded_original_circle_ids':[6,9,24,25],
    'v3_1_training_exclusion_buffer_m':15,
    'primary_modalities':['RGB','DSM-derived local geometry'],
    'optional_modalities':['MS bands/indices on verified survey overlaps','LAS SfM 3D diagnostics'],
    'ms_inventory_manifest':str(OUT/'00_ms_registry.json'),
    'ms_group_count':int(MS_META.group.nunique()),
    'ms_file_count':int(len(MS_META)),
    'ms_partial_coverage_policy':'Do not extrapolate YBJoseph/YBForgeo MS to Yangambi Carlier left; no Yoko MS',
    'comparison_support_policy':'Matched valid ROI∩RGB∩modality footprint; same frozen folds per comparison',
    'experimental_las':'small stratified crown sample, only if added information beyond DSM',
    'prohibited_unjustified_features':['CHM','DTM','understory profile from SfM'],
    'output_root':str(ROOT/'Outputs/v4'),
    'n_indexed_rasters':int((inventory.type=='raster').sum()),
    'n_indexed_las':int((inventory.type=='LAS').sum()),
    'rgb_dsm_pairings_confirmed':bool(len(pair_df)==len(RGB_NAMES) and (pair_df.rgb_matches.eq(1)&pair_df.dsm_matches.eq(1)&pair_df.crs_match).all()),
    'rgb_dsm_extent_qa_pass':bool(pair_df.extent_qa_pass.all()),
    'rgb_dsm_registration_verified':False,
    'next_step':'N01 after RGB–DSM identity, coverage, registration and dates pass QA',
}
manifest=OUT/'00_v4_input_audit_manifest.json'
manifest.write_text(json.dumps(audit,indent=2))
print('WROTE:',manifest)
print('RGB–DSM filenames/CRS confirmed:',audit['rgb_dsm_pairings_confirmed'])
print('RGB–DSM extent QA:',audit['rgb_dsm_extent_qa_pass'])
print('Subpixel registration still pending:',not audit['rgb_dsm_registration_verified'])
print('V3_1 remains unchanged:',V31_MANIFEST)
if not audit['rgb_dsm_pairings_confirmed']:
    print('ACTION REQUIRED: investigate unmatched filenames/CRS; check 00_rgb_dsm_pair_audit.csv.')
